In [ ]:
# Needs
# pip install fabric # This is for server connection
# pip install chardet # This is only for encoding errors. Not always necessary?
# Place initialization.py, jupyter_interface.py, cif_file, and this jupyter_notebook in the same location

In [ ]:
# Testing purposes. Updates class without having to restart kernel every time a change is made to source code.
%load_ext autoreload
%autoreload 2

In [ ]:
from jupyter_interface import *

In [ ]:
# If options are left as None/False the program will either not include those values, or it will automatically pick
# values that it deems appropriate. Most should be left to None unless there is a specific reason to change a value.

# All flags are True/False values
init_lapw_flags = {"-h":False, "-m":False, "-b":True, "-sp":False, "-nodstart":False, "-nokshift":False,
                    "-nometal":False, "-hdlo":False, "-nohdlo":False}

# Options are usually None/X, but can be True/False or None/[XYZ]
init_lapw_options = {"-f":None, "-prec":2, "-red":None, "-vxc":None, "-fftfac":None, "-fft":None, "-autofft":False,
                    "-ecut":None, "-rkmax":None, "-lmax":None, "-lvns":None, "-gmax":None, "-fermit":None,
                    "-fermits":None, "-numk":None, "-s":None, "-e":None}
# Note: "-fft":[X,Y,Z] is the valid input
# Note: "-autofft":True/False is the valid input

# Most slurm schedulers require an --account or --partition name. The rest can be left as None if desired.
# "misc":["--cores-per-socket=10","--norequeue"] This is a wildcard operator. Include the full commands as a string in a list
# that need to be added to the SBATCH commands if not present.
# If you add mail-user, also add mail-type
# Max cpus is not a default slurm parameter, but is used to determine the maxixum possible --ntasks-per-node
# Most clusters have 32 cpus per node, but if yours has more or less, adjust this value. Do not put None
# You must set both --nodes and --ntasks-per-node for it to take your set values.
# The auto selected ntasks may not be the fastest, but will always be computationally efficent.
# Max-mem-per-cpu dictates how much memory you want to allocate per cpu. Depends on cluster. Not slurm command.
# Default 3.9GB is usually good for high estimate, but should be set as low as possible to save resources.
# See https://slurm.schedmd.com/sbatch.html for full list of slurm commands
slurm_options = {"--job-name":None, "--mail-user":None, "--mail_type":None, "--account":None, "--partition":None,
                    "--nodes":None, "--ntasks-per-node":None, "--mem":None, "--time":None,
                    "max-ntasks-per-node":32, "max-mem-per-cpu":3.9, "misc":[]}

In [ ]:
from jupyter_interface import *

# Create an instance of the class
Structure = JupyterInterface(cif_file="TiCv2.cif")
# create_new_calculation will setup parameters to be used for calculations on server. Can have as many in a row as desired. Each will be run individually.
# Does not submit to server
# Press Shift+Tab on 'JupyterInferface' to see available parameters

# Because everything is a dictionary only one key value pair can exist. This means we can combine dictionaries together, giving precedence to one over the other.
# Insert them as **{**first, **priority}
# Can be expanded as far as you want if you want to organize them into different dictionaries. Highest number below has the highest priority in overwriting values
# Such as **{**{**{**first, **second}, **third}, **fourth}

# These are parameters you want to include in all calculations.
init_lapw_parameters = {
    "-nodstart":True, "-prec":3
}
slurm_parameters = {
    "xspec_elements":{"C":"1s","N":"1s","Ti":"2p","O":"1s","Fe":"3d"}
}

# These are parameters that you change between calculations.
# Define as empty dictionary to remove any previously stored values.
single_calc_parameters = {
    "-numk":600
}
Structure.create_new_calculation(**{**{**init_lapw_parameters, **slurm_parameters}, **single_calc_parameters})

single_calc_parameters = {
    "-numk":1200
}
Structure.create_new_calculation(**{**{**init_lapw_parameters, **slurm_parameters}, **single_calc_parameters})

# Functions if you want to interact with server
working_directory = 'ADD WORKING DIRECTORY ON CLUSTER' # Directory on the server you wish to run calculation
server_name = 'ADD SERVER ID host@server:port' # server login. If no ssh key then include a password below
# Usually if you have an ssh_key then password should be set to None, and vice versa
# If you set password to none you may still be prompted to enter a password each time you access the cluster
ssh_key = None # Options: None, True (if keys handled by server), or String location to ssh_key file
password = None # Options: None, String of your password

Structure.initialize_server(working_directory, server_name, ssh_key, password) # Sets up parameters to create a connection to server
Structure.submit_calculations() # Runs calculations as specified from .create_new_calculation (slurm submission turned off currently. Change into initialization.py to turn back on)
#Structure.download_info(do_hash=True) # Downloads info and stores it locally. File containing all info, and an hdf5.
#Structure.print_h5_structure() # Print the hdf5 of the same case name (needs downloaded info to function.

In [ ]:
# Only used if an encoding error happened during runtime. Will crash otherwise.
Structure.find_encoding()